# Jina CLIP v2 — 잔여분 임베딩 (Modal 92.7% 이후 보충)

Modal에서 $30 크레딧 한도로 중단된 잔여분 **48,965개**를 Kaggle 무료 T4 GPU로 처리.

- 진짜 미처리: 45,605개
- 이전 실패 재시도: 3,360개 (75%가 timeout/5xx/403 — Kaggle 다른 IP에서 복구 가능)

**사용 전 체크리스트**
1. 우측 패널 `Settings`:
   - **Accelerator**: `GPU T4 x2` 선택 (또는 P100)
   - **Internet**: `On` 토글 (이미지 다운로드 + worker POST 필요)
2. 좌측 `+ Add Data` → `Upload` → `remaining.jsonl` 업로드 (Dataset name: `remaining-items`)
3. 위에서 아래로 셀 순서대로 실행

Colab도 동일하게 동작 — `/content/remaining.jsonl`에 업로드.

## 실시간 모니터링
셀 4가 매 5배치마다 진행률 + ETA 출력 → 노트북 탭만 열어두면 바로 보임. `tqdm` 진행률 바도 같이 표시됨.


In [ ]:
# 1) 의존성 설치 — Kaggle pip이 4.45.2 정확히 못 찾는 경우 있어서 범위 지정
!pip install -q --upgrade 'transformers>=4.40,<5' einops timm pillow requests tqdm 2>&1 | tail -5
import transformers
print(f'✓ transformers {transformers.__version__} 설치됨')


In [ ]:
# 2) remaining.jsonl 위치 자동 탐색 + 못 찾으면 환경 상태 출력
import os, glob

candidates = [
    '/kaggle/input/remaining-items/remaining.jsonl',
    '/kaggle/working/remaining.jsonl',
    '/content/remaining.jsonl',
    './remaining.jsonl',
]
candidates += glob.glob('/kaggle/input/*/remaining.jsonl')
candidates += glob.glob('/kaggle/input/*/*.jsonl')  # 다른 파일명일 경우도 탐색

REMAINING_PATH = next((p for p in candidates if os.path.exists(p)), None)

if not REMAINING_PATH:
    print('❌ remaining.jsonl을 못 찾았어. 현재 상태:')
    print()
    print('=== /kaggle/input/ 안의 데이터셋 폴더 ===')
    if os.path.exists('/kaggle/input'):
        for d in os.listdir('/kaggle/input'):
            sub = f'/kaggle/input/{d}'
            try:
                files = os.listdir(sub)
                print(f'  {sub}/')
                for ff in files[:10]:
                    sz = os.path.getsize(f'{sub}/{ff}') / 1024 / 1024
                    print(f'    {ff}  ({sz:.1f} MB)')
            except: pass
    else:
        print('  /kaggle/input 이 없음 — Kaggle 환경 아님')
    print()
    print('=== /kaggle/working/ 안의 파일 ===')
    if os.path.exists('/kaggle/working'):
        for ff in os.listdir('/kaggle/working')[:10]:
            sz = os.path.getsize(f'/kaggle/working/{ff}') / 1024 / 1024
            print(f'  {ff}  ({sz:.1f} MB)')
    print()
    print('해결책:')
    print('  방법 A) 우측 패널 + Add Data > Upload 로 remaining.jsonl 업로드')
    print('         Dataset name: remaining-items (또는 아무 이름)')
    print('  방법 B) 좌측 파일 패널에서 /kaggle/working/ 로 직접 drag&drop 업로드')
    raise AssertionError('remaining.jsonl 없음 — 위 방법으로 업로드 후 이 셀 다시 실행')

print(f'✓ found: {REMAINING_PATH}')
size_mb = os.path.getsize(REMAINING_PATH) / 1024 / 1024
print(f'  size: {size_mb:.1f} MB')


In [ ]:
# 3) 설정 + Jina CLIP v2 로드 (~3GB 다운로드, 첫 실행만 ~2분)
import torch, json
from transformers import AutoModel

MODEL_ID = 'jinaai/jina-clip-v2'
WORKER_UPSERT_URL = 'https://armin-semantic-search.armin-art.workers.dev/upsert-jina'
CHECKPOINT_PATH = '/kaggle/working/kaggle_processed.txt'
if not os.path.exists('/kaggle'): CHECKPOINT_PATH = '/content/kaggle_processed.txt'

BATCH_SIZE = 32           # T4 16GB 적정 (Jina v2 EVA-L)
FETCH_WORKERS = 32        # 이미지 병렬 다운로드 수
FETCH_TIMEOUT = 30        # 단일 이미지 timeout (초) — Modal은 15였는데 timeout 실패 54%라 늘림

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

print(f'\nLoading {MODEL_ID}...')
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(device).eval()
if device == 'cuda':
    model = model.half()  # fp16으로 속도 2배
print('✓ model ready')


In [ ]:
# 4) 메인 임베딩 루프 — tqdm 진행률 + 매 5배치마다 통계 출력
import requests, io, time, urllib3
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from datetime import timedelta
from tqdm.auto import tqdm

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept': 'image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8',
}

# 체크포인트 (재실행 시 이어감)
processed = set()
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        for line in f:
            s = line.strip()
            if s: processed.add(s)
print(f'checkpoint: {len(processed):,} 이미 처리됨 (이 노트북 세션에서)')

# pending 로드 + skip processed
pending = []
with open(REMAINING_PATH) as f:
    for line in f:
        it = json.loads(line)
        if str(it['id']) in processed: continue
        pending.append(it)
print(f'pending: {len(pending):,}')

if not pending:
    print('✓ 모두 완료 — 다음 셀로')
else:
    sess = requests.Session()
    total_ok, total_bad, upsert_fail = 0, 0, 0
    t0 = time.time()

    def fetch(item):
        try:
            r = sess.get(item['i'], timeout=FETCH_TIMEOUT, headers=HEADERS, verify=False)
            r.raise_for_status()
            return item, Image.open(io.BytesIO(r.content)).convert('RGB'), None
        except Exception as e:
            return item, None, str(e)[:120]

    batches = [pending[i:i+BATCH_SIZE] for i in range(0, len(pending), BATCH_SIZE)]
    print(f'{len(batches):,} batches × {BATCH_SIZE} = {len(pending):,} items\n')

    pbar = tqdm(total=len(pending), desc='embedding', unit='img', smoothing=0.1)
    for bi, batch in enumerate(batches):
        with ThreadPoolExecutor(max_workers=FETCH_WORKERS) as ex:
            results = list(ex.map(fetch, batch))

        good = [(it, img) for it, img, err in results if img is not None]
        bad = [{'id': str(it['id']), 'e': str(it.get('e','')), 'error': err}
               for it, _, err in results if err is not None]

        if good:
            with torch.no_grad():
                vecs = model.encode_image([g[1] for g in good]).tolist()
            for _, img in good:
                try: img.close()
                except: pass

            payload = [
                {'id': str(it['id']), 'values': v, 'metadata': {'e': str(it.get('e',''))}}
                for (it, _), v in zip(good, vecs)
            ]
            try:
                r = sess.post(WORKER_UPSERT_URL, json={'vectors': payload}, timeout=30)
                if r.status_code == 200:
                    with open(CHECKPOINT_PATH, 'a') as f:
                        for it, _ in good:
                            f.write(str(it['id']) + '\n')
                    total_ok += len(good)
                else:
                    upsert_fail += 1
                    pbar.write(f'  [batch {bi}] upsert HTTP {r.status_code}: {r.text[:160]}')
            except Exception as e:
                upsert_fail += 1
                pbar.write(f'  [batch {bi}] upsert exception: {e}')

        if bad:
            with open(CHECKPOINT_PATH, 'a') as f:
                for b in bad:
                    f.write(b['id'] + '\n')
            total_bad += len(bad)

        pbar.update(len(batch))
        pbar.set_postfix({'ok': total_ok, 'bad': total_bad, 'ufail': upsert_fail})

        if (bi + 1) % 5 == 0:
            elapsed = time.time() - t0
            done = total_ok + total_bad
            rate = done / elapsed if elapsed > 0 else 0
            remaining_items = len(pending) - done
            eta = remaining_items / rate if rate > 0 else 0
            pbar.write(f'[{bi+1:>4}/{len(batches)}] {rate:.1f} imgs/s | ETA {str(timedelta(seconds=int(eta)))} | ok={total_ok:,} bad={total_bad:,}')

    pbar.close()
    print(f'\n=== DONE ===')
    print(f'ok: {total_ok:,}')
    print(f'bad (이미지 404 등): {total_bad:,}')
    print(f'upsert_fail: {upsert_fail}')
    print(f'총 소요: {timedelta(seconds=int(time.time()-t0))}')


In [ ]:
# 5) 검증 — Vectorize에 잘 들어갔는지 한 개 ID로 round-trip 체크
import random

ok_ids = []
with open(CHECKPOINT_PATH) as f:
    for line in f:
        s = line.strip()
        if s: ok_ids.append(s)

if ok_ids:
    sample = random.choice(ok_ids)
    print(f'sample ID: {sample}')
    r = sess.post('https://armin-semantic-search.armin-art.workers.dev/recommend-by-id',
                  json={'id': sample, 'limit': 3}, timeout=30)
    print(f'recommend status: {r.status_code}')
    print(r.text[:500])
else:
    print('체크포인트가 비어있음 — 셀 4가 실행됐는지 확인')